# qufin Technical Deep Dive

**For quant teams**: full QAOA circuit construction, QUBO math, QAE convergence, noise resilience.

1. QUBO formulation: Markowitz to Ising
2. QAOA circuit walkthrough with mixer comparison
3. QAE variants: convergence analysis
4. Noise resilience: how deep can circuits go?
5. Encoding comparison: qubit cost vs solution quality

In [ ]:
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## 1. QUBO Formulation: Markowitz to Quantum

The Markowitz objective `min gamma * w'Cov*w - mu'w` subject to `||w||_0 <= K` maps to a QUBO matrix `Q` where binary variable `x_i=1` means asset `i` is selected.

**Penalty terms stack**: return + risk + cardinality + sector + turnover.

In [ ]:
from qufin.portfolio.qubo import PortfolioQUBO

# 6-asset problem with cardinality + sector constraints
rng = np.random.default_rng(42)
n = 6
mu = rng.normal(0.0003, 0.0001, n)
cov = np.cov(rng.normal(0, 0.01, (252, n)), rowvar=False)

qubo = PortfolioQUBO(
    mu=mu, cov=cov, gamma=1.0,
    cardinality=3,
    sector_map={0: [0, 1, 2], 1: [3, 4, 5]},
    sector_caps={0: 2, 1: 2},
    budget_penalty=1e4,
    encoding="one_hot",
)
Q = qubo.build_matrix()

print(f"QUBO matrix: {Q.shape[0]}x{Q.shape[1]}")
print(f"Non-zero entries: {np.count_nonzero(Q)}")
print(f"Density: {np.count_nonzero(Q) / Q.size:.1%}")
print(f"\nDiagonal (return - penalty terms):")
for i in range(n):
    print(f"  Asset {i}: Q[{i},{i}] = {Q[i,i]:>12.4f}")

print(f"\nOff-diagonal sample (risk + penalty coupling):")
for i in range(min(3, n)):
    for j in range(i+1, min(4, n)):
        if abs(Q[i,j]) > 1e-10:
            print(f"  Q[{i},{j}] = {Q[i,j]:>12.4f}")

In [ ]:
# Exhaustive solve — verify optimal solution
from qufin.portfolio.optimizers.exhaustive import exhaustive_solve

exact = exhaustive_solve(qubo, return_all=True)
print(f"Optimal bitstring: {exact.best_bitstring}")
print(f"Optimal objective: {exact.best_objective:.6f}")
print(f"Selected assets: {[i for i, b in enumerate(exact.best_bitstring) if b == '1']}")
print(f"Weights: {exact.weights}")
print(f"Feasible: {exact.feasible}")
print(f"States evaluated: {exact.n_evaluated:,}")

# Show top 5 solutions
objectives = np.array(exact.all_objectives)
top_idx = np.argsort(objectives)[:5]
print(f"\nTop 5 solutions:")
for rank, idx in enumerate(top_idx):
    bs = format(idx, f'0{n}b')
    feas = qubo.feasibility_check(bs)
    print(f"  #{rank+1}: {bs} obj={objectives[idx]:>12.4f} feas={feas}")

## 2. QAOA Mixer Comparison

Compare X (unconstrained), XY-ring (Hamming-preserving), and XY-full mixers on the same problem.

In [ ]:
from qufin.portfolio.optimizers.qaoa import QAOAPortfolio, QAOAConfig
from qufin.backends.qiskit_backend import QiskitAerBackend

backend = QiskitAerBackend(seed=42)
optimal_obj = exact.best_objective

print(f"Optimal objective: {optimal_obj:.6f}")
print(f"\n{'Mixer':<12s} | {'p':>2s} | {'Best Obj':>14s} | {'Approx Ratio':>12s} | {'Feasible':>8s} | {'Time':>7s}")
print("-" * 72)

for mixer in ["x", "xy_ring", "xy_full"]:
    for p in [1, 2, 3]:
        card = 3 if mixer != "x" else None
        cfg = QAOAConfig(p=p, mixer=mixer, cardinality=card,
                         shots=4096, seed=42, maxiter=80)
        t0 = time.perf_counter()
        res = QAOAPortfolio(qubo, cfg, backend).run()
        elapsed = time.perf_counter() - t0
        ratio = res.best_objective / optimal_obj if optimal_obj != 0 else 0
        print(f"{mixer:<12s} | {p:>2d} | {res.best_objective:>14.4f} | {ratio:>11.4f}x | "
              f"{str(res.feasible):>8s} | {elapsed:>6.1f}s")

## 3. Circuit Depth Analysis

Transpiled circuit depth determines NISQ feasibility. XY mixers are deeper but preserve constraints.

In [ ]:
from qufin.portfolio.mixers import get_mixer, DickeInitialState
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

pm = generate_preset_pass_manager(optimization_level=2, basis_gates=["cx", "rz", "sx", "x"])

print(f"{'Qubits':>6s} | {'Mixer':<10s} | {'Raw Depth':>10s} | {'Transp Depth':>12s} | {'CX Count':>8s}")
print("-" * 58)

for nq in [4, 6, 8, 10, 12]:
    k = max(2, nq // 3)
    for mixer_name in ["x", "xy_ring", "xy_full"]:
        qc = QuantumCircuit(nq)
        if mixer_name in ("xy_ring", "xy_full"):
            init = DickeInitialState(nq, k)
            qc.compose(init.circuit(), inplace=True)
        mixer = get_mixer(mixer_name, nq, cardinality=k if mixer_name != "x" else None)
        qc.compose(mixer.circuit(beta=0.5), inplace=True)

        transpiled = pm.run(qc)
        cx = transpiled.count_ops().get("cx", 0)
        print(f"{nq:>6d} | {mixer_name:<10s} | {qc.depth():>10d} | {transpiled.depth():>12d} | {cx:>8d}")

## 4. QAE Convergence Analysis

Compare all 4 QAE variants on a known amplitude `a = sin^2(pi/6) = 0.25`.

In [ ]:
from qufin.options.amplitude_estimation.estimation_problem import EstimationProblem
from qufin.options.amplitude_estimation.canonical import CanonicalAmplitudeEstimation, CanonicalQAEConfig
from qufin.options.amplitude_estimation.iqae import IterativeAmplitudeEstimation, IQAEConfig

true_a = 0.25
A = QuantumCircuit(1)
A.ry(2 * np.arcsin(np.sqrt(true_a)), 0)
problem = EstimationProblem(state_preparation=A, objective_qubits=[0])

print(f"True amplitude: a = sin^2(pi/6) = {true_a}")
print(f"\n{'Variant':<25s} | {'Config':>15s} | {'Estimate':>10s} | {'Abs Error':>10s} | {'Time':>7s}")
print("-" * 78)

# Canonical QAE — increasing eval qubits
for nq in [3, 4, 5, 6, 7, 8]:
    cfg = CanonicalQAEConfig(n_eval_qubits=nq, shots=4096, seed=42)
    t0 = time.perf_counter()
    res = CanonicalAmplitudeEstimation(problem, cfg, backend).estimate()
    t = time.perf_counter() - t0
    err = abs(res.estimation - true_a)
    print(f"Canonical QAE             | {nq:>8d} qubits | {res.estimation:>10.6f} | {err:>10.6f} | {t:>6.3f}s")

print()

# IQAE — decreasing epsilon
for eps in [0.1, 0.05, 0.01, 0.005]:
    cfg = IQAEConfig(epsilon=eps, alpha=0.05, shots=4096, seed=42)
    t0 = time.perf_counter()
    res = IterativeAmplitudeEstimation(problem, cfg, backend).estimate()
    t = time.perf_counter() - t0
    err = abs(res.estimation - true_a)
    print(f"IQAE                      | {'eps='+str(eps):>15s} | {res.estimation:>10.6f} | {err:>10.6f} | {t:>6.3f}s")

## 5. Noise Resilience

Sweep noise levels and observe how QAOA solution quality degrades. This determines the maximum useful circuit depth on real hardware.

In [ ]:
from qufin.backends.noise_models import NoisyAerBackend, NoiseProfile, IDEAL

# 4-asset QAOA under increasing noise
small_mu = mu[:4]
small_cov = cov[:4, :4]
small_qubo = PortfolioQUBO(mu=small_mu, cov=small_cov, gamma=1.0,
                           cardinality=2, budget_penalty=1e4)
small_exact = exhaustive_solve(small_qubo)
optimal = small_exact.best_objective

error_rates = [0, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2]
cfg = QAOAConfig(p=1, mixer="xy_ring", cardinality=2, shots=4096, seed=42, maxiter=50)

print(f"Optimal objective: {optimal:.6f}")
print(f"\n{'2Q Error':>10s} | {'Best Obj':>14s} | {'Approx Ratio':>12s} | {'Feasible':>8s}")
print("-" * 55)

for rate in error_rates:
    if rate == 0:
        profile = IDEAL
    else:
        profile = NoiseProfile(
            single_gate_error=rate/10, two_gate_error=rate,
            readout_error=min(rate*3, 0.15), name=f"sweep_{rate}",
        )
    noisy = NoisyAerBackend(profile=profile, seed=42)
    res = QAOAPortfolio(small_qubo, cfg, noisy).run()
    ratio = res.best_objective / optimal if optimal != 0 else 0
    print(f"{rate:>10.1e} | {res.best_objective:>14.4f} | {ratio:>11.4f}x | {str(res.feasible):>8s}")

## 6. Encoding Comparison

One-hot vs binary encoding: trade-off between qubit count and solution granularity.

In [ ]:
from qufin.portfolio.encodings import qubit_cost_table

table = qubit_cost_table([5, 10, 20, 50, 100, 200])

print(f"{'Assets':>7s} | {'One-Hot':>8s} | {'Binary-3':>9s} | {'Binary-5':>9s} | {'Unary-4':>8s}")
print("-" * 52)
for row in table:
    print(f"{row['n_assets']:>7d} | {row['one_hot']:>8d} | {row['binary_3bit']:>9d} | "
          f"{row['binary_5bit']:>9d} | {row['unary_4level']:>8d}")

print("\nKey insight: One-hot is most qubit-efficient for selection problems.")
print("Binary encoding allows finer weight granularity but costs 3-5x more qubits.")
print("NISQ devices (100-150 qubits) can handle ~100 assets with one-hot,")
print("but only ~30-50 with binary encoding.")

## 7. Grover Operator Correctness

The Grover operator eigenvalues must be `exp(+/- 2i*theta)`. Without the global phase `-1` (Brassard's minus sign), eigenvalues become `exp(+/- 4i*theta)`, breaking all QAE variants.